In [2]:
import pandas as pd
import numpy as np

# --- PART A: Data Setup ---

# A1. Load dataset (UCI Abalone Dataset) directly from the web
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
column_names = ["Sex", "Length", "Diameter", "Height", "Whole weight",
                "Shucked weight", "Viscera weight", "Shell weight", "Rings"]
data = pd.read_csv(url, names=column_names)

print(f"Number of rows: {len(data)}")
print(f"Column names: {data.columns.tolist()}")
print("First 5 rows:\n", data.head())

# Checkpoint:
# what is input: Numeric measurements like Length, Diameter, and Weight.
# what is output: The age of the abalone.
# why output is numeric: Because we are predicting a continuous value (age), not a category.

# A2. Convert target: y = Rings + 1.5
y = (data["Rings"] + 1.5).values.reshape(-1, 1)

# A3. Choose exactly 3 numeric features
# Justification:
# Feature 1: Length - Base size of the shell.
# Feature 2: Diameter - Important for overall growth volume.
# Feature 3: Whole weight - Heaviness is a strong indicator of age in biological samples.
X = data[["Length", "Diameter", "Whole weight"]].values

# A4. Train-test split (80/20) without sklearn
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# A5. Normalize inputs (Mandatory for Gradient Descent)
train_mean = np.mean(X_train, axis=0)
train_std = np.std(X_train, axis=0)
X_train_norm = (X_train - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

# Checkpoint:
# why normalization is needed: It prevents features with large scales (like weight)
# from dominating the gradients, making the training much faster and stable.

# --- PART B: Define the Model ---
def forward(X, w, b):
    # Model equation: y_hat = X.w + b
    return np.dot(X, w) + b

# Initializing weights with small random numbers and bias with zero
d = X_train_norm.shape[1]
w = np.random.randn(d, 1) * 0.01
b = 0.0

# Checkpoint:
# parameters are: weights (w1, w2, w3) and a scalar bias (b).
# number of parameters: 4

# --- PART C: Define Loss (MSE) ---
def mse(y, y_hat):
    return np.mean((y - y_hat)**2)

# Checkpoint:
# why square: To make all errors positive and penalize larger errors more heavily.

# --- PART D: Learning Rule (Gradients) ---
def grad_w(X, y, y_hat):
    N = len(y)
    return (2/N) * np.dot(X.T, (y_hat - y))

def grad_b(y, y_hat):
    N = len(y)
    return (2/N) * np.sum(y_hat - y)

# Checkpoint:
# what gradient means: It is the slope that tells us the direction to reduce error.
# why subtracting reduces loss: Moving opposite to the gradient goes downhill towards the minimum loss.

# --- PART E: Training Loop ---
lr = 0.01
epochs = 1000

print("\n--- Starting Training ---")
for epoch in range(epochs):
    # 1) Forward Pass
    y_hat = forward(X_train_norm, w, b)

    # 2) Compute Loss
    loss = mse(y_train, y_hat)

    # 3) Compute Gradients
    dw = grad_w(X_train_norm, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    # 4) Update Parameters
    w -= lr * dw
    b -= lr * db

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: Loss = {loss:.4f}")

# --- PART F: Evaluation ---
y_test_pred = forward(X_test_norm, w, b)
test_mse = mse(y_test, y_test_pred)
test_mae = np.mean(np.abs(y_test - y_test_pred))

print(f"\nFinal Test MSE: {test_mse:.4f}")
print(f"Final Test MAE: {test_mae:.4f}")

print("\nSample Predictions (True vs Predicted):")
for i in range(5):
    t = y_test[i][0]
    p = y_test_pred[i][0]
    print(f"True Age: {t:.2f} | Predicted: {p:.2f} | Abs Error: {abs(t-p):.2f}")

Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']
First 5 rows:
   Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  

--- Starting Training ---
Epoch 0: Loss = 144.4062
Epoch 200: Loss = 7.5028
Epoch 400: Loss = 7.4447
Epoch 600: Loss = 7.4304
Epoch 800: Loss = 7.4180

Final Test MSE: 5.3514
Final Test MAE: 1.80